# 📡 Telco Customer Churn Prediction — CBSOT Summer Internship 2026
## Notebook 1: Exploratory Data Analysis & Preprocessing

**Dataset:** IBM Telco Customer Churn Dataset  
**Records:** 7,043 customers | 33 columns  
**Goal:** Understand customer behavior and prepare data for ML modeling

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Create outputs folder if not exists
os.makedirs('../outputs', exist_ok=True)

# Plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('Libraries imported successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_excel('../data/Telco_customer_churn.xlsx')

print(f'Dataset Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head()

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
print('=== Numerical Summary ===')
df[['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Value']].describe().round(2)

## 3. Target Variable Analysis

In [ ]:
# Class distribution
churn_counts = df['Churn Label'].value_counts()
churn_pct    = df['Churn Label'].value_counts(normalize=True) * 100

print('=== Churn Distribution ===')
for label in churn_counts.index:
    print(f'  {label}: {churn_counts[label]:,} customers ({churn_pct[label]:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Countplot
sns.countplot(x='Churn Label', data=df, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn Label')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', 
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(churn_counts, labels=churn_counts.index, autopct='%1.2f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90,
            textprops={'fontsize': 12})
axes[1].set_title('Churn Rate Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/01_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: outputs/01_churn_distribution.png')

## 4. EDA — Numerical Features

In [ ]:
# Tenure Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
sns.histplot(df['Tenure Months'], bins=30, kde=True, ax=axes[0], color='#3498db')
axes[0].set_title('Tenure Months Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tenure (Months)')

# Churn vs Tenure
sns.boxplot(x='Churn Label', y='Tenure Months', data=df, ax=axes[1],
            palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Tenure vs Churn', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/02_tenure_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Key insight
churned_tenure    = df[df['Churn Label'] == 'Yes']['Tenure Months'].mean()
retained_tenure   = df[df['Churn Label'] == 'No']['Tenure Months'].mean()
print(f'\n📊 Key Insight — Average Tenure:')
print(f'  Churned customers  : {churned_tenure:.2f} months')
print(f'  Retained customers : {retained_tenure:.2f} months')
print(f'  Difference         : {retained_tenure - churned_tenure:.2f} months')

In [ ]:
# Monthly Charges Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Monthly Charges'], bins=30, kde=True, ax=axes[0], color='#9b59b6')
axes[0].set_title('Monthly Charges Distribution', fontsize=13, fontweight='bold')

sns.boxplot(x='Churn Label', y='Monthly Charges', data=df, ax=axes[1],
            palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Monthly Charges vs Churn', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/03_monthly_charges_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Quartile analysis — churned vs retained separately
print('\n📊 Monthly Charges Quartile Analysis:')
print('\n  CHURNED customers:')
print(df[df['Churn Label'] == 'Yes']['Monthly Charges'].describe().round(2))
print('\n  RETAINED customers:')
print(df[df['Churn Label'] == 'No']['Monthly Charges'].describe().round(2))

## 5. EDA — Categorical Features

In [ ]:
# Contract Type Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x='Contract', hue='Churn Label', data=df, ax=axes[0],
              palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Contract Type vs Churn', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Contract Type')
axes[0].legend(title='Churn')

# Churn rate per contract
contract_churn = pd.crosstab(df['Contract'], df['Churn Label'], normalize='index') * 100
contract_churn['Yes'].plot(kind='bar', ax=axes[1], color=['#e74c3c', '#f39c12', '#2ecc71'],
                            edgecolor='black')
axes[1].set_title('Churn Rate by Contract Type (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Contract Type')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].tick_params(axis='x', rotation=15)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/04_contract_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Churn Rate by Contract Type:')
print(contract_churn['Yes'].round(2).to_string())

In [ ]:
# Internet Service, Payment Method, Tech Support
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['Internet Service', 'Payment Method', 'Tech Support']):
    sns.countplot(x=col, hue='Churn Label', data=df, ax=ax,
                  palette=['#2ecc71', '#e74c3c'])
    ax.set_title(f'{col} vs Churn', fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(title='Churn', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/05_categorical_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation Analysis

In [ ]:
num_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Value', 'Churn Score', 'CLTV']
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size': 11})
plt.title('Correlation Heatmap — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Correlation with Churn Value:')
print(corr_matrix['Churn Value'].sort_values(ascending=False).round(3).to_string())

## 7. Data Preprocessing

In [ ]:
# Step 1: Fix Total Charges dtype
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
print(f'Missing in Total Charges: {df["Total Charges"].isnull().sum()}')

# Step 2: Fill missing (0-tenure customers have no total charges)
df['Total Charges'].fillna(0, inplace=True)
print('Missing values filled with 0 (0-tenure customers)')

In [ ]:
# Step 3: Drop irrelevant / leakage columns
drop_cols = [
    'CustomerID', 'Count', 'Country', 'State', 'City',
    'Zip Code', 'Lat Long', 'Latitude', 'Longitude',
    'Churn Label', 'Churn Score', 'CLTV', 'Churn Reason'
]
df_clean = df.drop(columns=drop_cols)
print(f'Columns after dropping: {df_clean.shape[1]}')
print(f'Remaining columns: {list(df_clean.columns)}')

In [ ]:
# Step 4: One-Hot Encoding
df_encoded = pd.get_dummies(df_clean, drop_first=True)
print(f'Shape after encoding: {df_encoded.shape}')
print(f'Sample encoded columns: {list(df_encoded.columns[:10])}')

In [ ]:
# Step 5: Define X and y
X = df_encoded.drop('Churn Value', axis=1)
y = df_encoded['Churn Value']

print(f'Feature matrix X: {X.shape}')
print(f'Target vector y: {y.shape}')
print(f'\nClass distribution in y:')
print(y.value_counts())
print(f'\nClass balance: {y.value_counts(normalize=True).round(4) * 100}')

# Save for notebook 2
df_encoded.to_csv('../outputs/df_preprocessed.csv', index=False)
print('\n✅ Preprocessed data saved to outputs/df_preprocessed.csv')

## 8. Key EDA Summary

| Insight | Finding |
|---|---|
| **Class Imbalance** | 73.46% Retained vs 26.54% Churned |
| **Avg Tenure — Churned** | ~17.98 months |
| **Avg Tenure — Retained** | ~37.57 months |
| **Month-to-Month Churn Rate** | ~42.7% |
| **One-Year Contract Churn** | ~11.3% |
| **Two-Year Contract Churn** | ~2.8% |
| **Top Churn Driver** | Short tenure + high monthly charges + M2M contract |
| **Tech Support Impact** | Customers without tech support churn significantly more |

---
➡️ **Next:** Notebook 2 — Modeling, SHAP, XGBoost, Segmentation